# Neural Audio Codecs — EnCodec, SNAC, Mimi, DAC and the Semantic-Acoustic Split Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: encode with EnCodec

In [ ]:
```python

from encodec import EncodecModel

import torch

model = EncodecModel.encodec_model_24khz()

model.set_target_bandwidth(6.0)  # kbps

wav = torch.randn(1, 1, 24000)

with torch.no_grad():

    encoded = model.encode(wav)

codes, scale = encoded[0]

# codes: (1, n_codebooks, n_frames), dtype=int64

In [ ]:
```

`n_codebooks=8` at 6 kbps. Each code is 0-1023 (10-bit).

### Step 2: decode and measure reconstruction

In [ ]:
```python

with torch.no_grad():

    wav_recon = model.decode([(codes, scale)])

from torchaudio.functional import compute_deltas

import torch.nn.functional as F

mse = F.mse_loss(wav_recon[:, :, :wav.shape[-1]], wav).item()

In [ ]:
```

### Step 3: the semantic-acoustic split (Mimi-style)

In [ ]:
```python

from moshi.models import loaders

mimi = loaders.get_mimi()

with torch.no_grad():

    codes = mimi.encode(wav)  # shape (1, 8, frames@12.5Hz)

semantic = codes[:, 0]

acoustic = codes[:, 1:]

In [ ]:
```

Semantic codebook 0 is WavLM-aligned. You can train a text-to-semantic transformer — much smaller vocabulary than going direct-to-audio. Then a separate acoustic-to-waveform decoder conditions on a speaker reference.

### Step 4: why AR LM over codec tokens works

For a 10 s speech clip at Mimi's 12.5 Hz × 8 codebooks:

In [ ]:
```

N_tokens = 10 * 12.5 * 8 = 1000 tokens

In [ ]:
```

1000 tokens is a trivial context for a transformer. A 256M-parameter transformer can generate 10 seconds of speech in milliseconds on a modern GPU.

## Exercises

In [ ]:
1. **Easy.** Run `code/main.py`. It implements a toy scalar + residual quantizer and measures reconstruction error as you add codebooks.
2. **Medium.** Install `encodec` and compare 1, 4, 8, 32 codebooks on a held-out speech clip. Plot PESQ or MSE vs bitrate.
3. **Hard.** Load Mimi. Encode a clip. Replace codebook 0 with random integers; decode. Then replace codebook 7 similarly. Compare the two corruptions — codebook 0 corruption should destroy intelligibility; codebook 7 corruption should barely change anything.